### Ex1

In [ ]:
import csv
import re
import sys
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm

CEX_PATH = Path(r"lsj_chicago.cex")
OUT_PATH = Path(r"lsj_social_relations.csv")
OUT_SCORES = Path(r"lsj_social_relations_scores.csv")

EMBED_MODEL = "all-MiniLM-L6-v2"
SIM_THRESHOLD = 0.45
TEST_LIMIT = 0

CATEGORIES = {
    "kinship_family": {
        "seeds": [
            "father", "mother", "brother", "sister", "son", "daughter",
            "uncle", "aunt", "nephew", "niece", "cousin", "grandfather",
            "grandmother", "grandson", "kinsman", "kinswoman", "kindred",
            "kinship", "ancestor", "descendant", "offspring", "parent",
            "sibling", "twin", "orphan", "widow", "widower", "heir",
            "stepmother", "stepfather", "foster", "family", "clan",
            "from the same womb", "childless", "firstborn",
        ],
        "prompts": ["kinship, family relations, ancestry, siblings and lineage"],
    },
    "marriage": {
        "seeds": [
            "marriage", "married", "unmarried", "wedding", "bride",
            "bridegroom", "wife", "husband", "spouse", "betroth", "dowry",
            "wedlock", "concubine", "bachelor", "celibacy", "divorce",
            "wooing", "suitor",
        ],
        "prompts": ["marriage, weddings, husbands, wives and betrothal"],
    },
    "friendship_affection": {
        "seeds": [
            "friend", "friendship", "companion", "comrade", "beloved",
            "darling", "affection", "loving", "love of", "dear to",
            "fond of", "brotherly love", "goodwill", "benevolent",
        ],
        "prompts": [
            "friendship, companionship, affection and love between people",
            "goodwill, kindness and benevolence towards another person",
        ],
    },
    "hostility_enmity": {
        "seeds": [
            "enemy", "enmity", "hostile", "hostility", "hatred", "hate",
            "foe", "quarrel", "feud", "grudge", "rival", "rivalry",
            "envy", "jealous", "wrath", "revenge", "vengeance", "strife",
            "adversary", "insolence", "insolent", "arrogance", "abuse",
            "reproach", "insult",
        ],
        "prompts": [
            "hostility, enmity, hatred and being someone's enemy",
            "envy, jealousy, grudges, quarrels and rivalry between people",
        ],
    },
    "hospitality_xenia": {
        "seeds": [
            "guest", "host", "hospitality", "stranger", "guest-friend",
            "xenia", "welcome", "entertain", "banquet", "feast",
            "table-companion", "inhospitable",
        ],
        "prompts": ["hospitality, guest-friendship, welcoming strangers and guests"],
    },
    "erotic_pederasty": {
        "seeds": [
            "lover", "beloved boy", "pederasty", "erotic", "courtesan",
            "paramour", "mistress", "sexual love", "desire for", "wooer",
            "eromenos", "erastes", "hetaira",
        ],
        "prompts": ["erotic love, lovers, courtesans and pederastic relationships"],
    },
    "rank_kingship_rule": {
        "seeds": [
            "king", "queen", "ruler", "rule", "authority", "power",
            "prince", "queenly", "lord", "master", "despot", "tyrant",
            "chief", "leader", "govern", "office", "rank", "nobility",
            "aristocrat", "birth", "high-born", "low-born",
        ],
        "prompts": [
            "kingship, royalty, rulers, tyrants and political authority",
            "social rank, nobility, high or low birth, aristocracy",
        ],
    },
    "military_rank": {
        "seeds": [
            "general", "commander", "captain", "officer", "soldier",
            "troop", "army", "phalanx", "garrison", "recruit", "levy",
            "muster", "host-leading", "leader of the host",
            "comrade in arms", "brave", "valiant",
        ],
        "prompts": ["military ranks, commanders, generals and leading troops"],
    },
    "servitude_slavery": {
        "seeds": [
            "slave", "slavery", "servant", "serf", "servitude", "bondage",
            "freedman", "manumission", "handmaid", "attendant",
            "hired labourer", "hireling", "impress", "impressment",
            "forced service",
        ],
        "prompts": ["slavery, servants, servitude and forced labour"],
    },
    "messenger_herald_envoy": {
        "seeds": [
            "messenger", "envoy", "herald", "ambassador", "courier",
            "message", "tidings", "dispatch",
        ],
        "prompts": ["messengers, heralds, envoys and ambassadors between parties"],
    },
    "civic_community": {
        "seeds": [
            "citizen", "citizenship", "fellow-citizen", "assembly",
            "community", "tribe", "deme", "gregarious", "social",
            "band of youths", "age-class", "member of",
        ],
        "prompts": ["civic community, citizenship, tribes and organized groups of people"],
    },
}

GREEK_RANGES = ((0x0370, 0x03FF), (0x1F00, 0x1FFF))
BOLD_RE = re.compile(r"\*\*(.+?)\*\*", re.S)


def is_mostly_greek(text: str, ratio: float = 0.5) -> bool:
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return False
    greek = sum(1 for c in letters if any(lo <= ord(c) <= hi for lo, hi in GREEK_RANGES))
    return greek / len(letters) >= ratio


def extract_gloss(entry: str) -> str:
    spans = []
    for match in BOLD_RE.finditer(entry):
        chunk = match.group(1).strip()
        if not chunk or is_mostly_greek(chunk):
            continue
        spans.append(chunk)
    gloss = "; ".join(dict.fromkeys(spans))
    return re.sub(r"\s+", " ", gloss).strip("; ").strip()


def parse_cex(path: Path, limit: int = 0):
    count = 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split("#", 3)
            if len(parts) < 4:
                continue
            seq, urn, lemma, entry = parts
            if not urn.startswith("urn:"):
                continue
            yield seq, urn, lemma.strip("· "), entry
            count += 1
            if limit and count >= limit:
                break


def compile_seed_patterns():
    patterns = {}
    for cat, spec in CATEGORIES.items():
        alts = sorted(spec["seeds"], key=len, reverse=True)
        pat = r"\b(?:" + "|".join(re.escape(s) for s in alts) + r")\b"
        patterns[cat] = re.compile(pat, re.IGNORECASE)
    return patterns


def main():
    if not CEX_PATH.exists():
        sys.exit(f"ERROR: CEX file not found:\n  {CEX_PATH}")

    t0 = time.time()
    print(f"\n[1/4] Parsing {CEX_PATH.name} ...")
    records = []
    for seq, urn, lemma, entry in tqdm(parse_cex(CEX_PATH, limit=TEST_LIMIT), desc="  parsing", unit=" entries"):
        gloss = extract_gloss(entry)
        if gloss:
            records.append({"seq": seq, "urn": urn, "lemma": lemma, "gloss": gloss, "entry": entry})
    print(f"  → {len(records):,} entries with an English gloss  ({time.time()-t0:.1f}s)")

    print("\n[2/4] Seed-keyword pass ...")
    t1 = time.time()
    seed_patterns = compile_seed_patterns()
    seed_matched, unmatched = [], []
    for rec in records:
        hits = [cat for cat, pat in seed_patterns.items() if pat.search(rec["gloss"])]
        rec["seed_cats"] = hits
        (seed_matched if hits else unmatched).append(rec)
    print(f"  → {len(seed_matched):,} entries matched by seeds, {len(unmatched):,} remain for embedding  ({time.time()-t1:.1f}s)")

    print(f"\n[3/4] Stage 2: embedding similarity on {len(unmatched):,} unmatched entries ...")
    print(f"  Loading model '{EMBED_MODEL}' ...")
    t2 = time.time()

    import numpy as np
    from sentence_transformers import SentenceTransformer

    try:
        from huggingface_hub import snapshot_download

        local_path = snapshot_download(
            repo_id=f"sentence-transformers/{EMBED_MODEL}",
            token=None,
            ignore_patterns=["additional_chat_templates*", "*.msgpack", "flax_model*", "tf_model*", "rust_model*"],
        )
        model = SentenceTransformer(local_path)
    except Exception:
        model = SentenceTransformer(f"sentence-transformers/{EMBED_MODEL}")
    print(f"  Model loaded ({time.time()-t2:.1f}s)")

    cat_names, cat_mats = [], []
    for cat, spec in CATEGORIES.items():
        texts = spec["prompts"] + spec["seeds"]
        vecs = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        cat_names.append(cat)
        cat_mats.append(vecs)

    if unmatched:
        um_vecs = model.encode([r["gloss"] for r in unmatched], normalize_embeddings=True, batch_size=256, show_progress_bar=True)
        for i, rec in enumerate(unmatched):
            v = um_vecs[i]
            scores = {name: float(np.max(mat @ v)) for name, mat in zip(cat_names, cat_mats)}
            rec["embed_scores"] = scores
            rec["embed_cats"] = [c for c, s in scores.items() if s >= SIM_THRESHOLD]
    else:
        print("  (nothing to embed)")

    print(f"  Embedding stage done ({time.time()-t2:.1f}s)")
    for rec in seed_matched:
        rec["embed_scores"] = {c: 0.0 for c in cat_names}
        rec["embed_cats"] = []

    print("\n[4/4] Writing results ...")
    all_records = seed_matched + unmatched
    output_rows, score_rows = [], []

    for rec in all_records:
        cats = sorted(set(rec["seed_cats"]) | set(rec["embed_cats"]))
        method = "seed+embed" if rec["seed_cats"] and rec["embed_cats"] else "seed" if rec["seed_cats"] else "embed" if rec["embed_cats"] else "none"
        best_cat = max(rec["embed_scores"], key=rec["embed_scores"].get) if rec["embed_scores"] else ""
        best_score = rec["embed_scores"].get(best_cat, 0.0)

        score_rows.append(
            {
                "lemma": rec["lemma"],
                "gloss": rec["gloss"],
                "best_category": best_cat,
                "best_score": round(best_score, 3),
                **{f"score_{c}": round(rec["embed_scores"].get(c, 0.0), 3) for c in cat_names},
            }
        )

        if cats:
            for cat in cats:
                output_rows.append(
                    {
                        "lemma": rec["lemma"],
                        "urn": rec["urn"],
                        "category": cat,
                        "method": method,
                        "embed_score": round(rec["embed_scores"].get(cat, 0.0), 3),
                        "gloss": rec["gloss"],
                    }
                )

    df = pd.DataFrame(output_rows)
    scores_df = pd.DataFrame(score_rows)
    df.to_csv(OUT_PATH, index=False)
    scores_df.to_csv(OUT_SCORES, index=False)

    print(f"\n✓ Done in {(time.time()-t0)/60:.1f} min")
    print(f"  {len(df):,} category assignments ({df['lemma'].nunique():,} unique lemmata)")
    print(f"  Main CSV  → {OUT_PATH}")
    print(f"  Scores CSV→ {OUT_SCORES}")
    print("\nEntries per category:")
    print(df["category"].value_counts().to_string())


### Ex2

In [13]:
import csv
import re
import sys
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm


CEX_PATH   = Path(r"lsj_chicago.cex")
OUT_PATH   = Path(r"lsj_social_relations.csv")
OUT_SCORES = Path(r"lsj_social_relations_scores.csv")

EMBED_MODEL = "all-MiniLM-L6-v2"   # ~80 MB, cached after first download

# --- The precision knobs. Raise these -> fewer, tighter results. -----------
THRESHOLD             = 0.7   # a sense must reach this similarity to a positive prompt
SEED_CONFIRM_THRESHOLD = 0.40  # lower bar granted to senses whose word also hit a seed keyword
MARGIN                = 0.06   # a sense must beat its nearest DECOY by at least this much
MULTI_LABEL           = False  # False = each word -> its single best category (recommended)
TIE_DELTA             = 0.04   # if MULTI_LABEL: also keep categories within this of the top

# Set to a positive integer (e.g. 5000) for a quick test run, 0 = full corpus.
TEST_LIMIT  = 0


GLOBAL_ANTI_PROMPTS = [
    "love of money, greed for wealth, avarice",
    "love of honour, glory, or ambition",
    "love of power, rule, or victory",
    "love of wisdom, learning, or the arts",
    "fondness for food, drink, or bodily pleasure",
    "a plant, herb, tree, flower, or fruit",
    "an animal, bird, fish, reptile, or insect",
    "a part of the body or a bodily organ",
    "a disease, wound, or medical condition",
    "a technical term in grammar, rhetoric, music, or metre",
    "a coin, weight, measure, or unit of quantity",
    "a tool, vessel, instrument, or physical object",
    "a geographical place, river, mountain, or region",
    "an abstract quality, action, state, or grammatical form",
]


CATEGORIES = {
    "kinship_family": {
        "prompts": [
            "a family relationship such as father, mother, brother, or sister",
            "kinship, blood relatives, ancestry, and descent",
            "children, offspring, heirs, and orphans within a family",
        ],
        "anti_prompts": [
            "a member of a guild, party, or group unrelated to blood",
        ],
        "seeds": [
            "father", "mother", "brother", "sister", "son", "daughter",
            "uncle", "aunt", "nephew", "niece", "cousin", "grandfather",
            "grandmother", "grandson", "kinsman", "kinswoman", "kindred",
            "kinship", "ancestor", "descendant", "offspring", "parent",
            "sibling", "twin", "orphan", "widow", "widower", "heir",
            "stepmother", "stepfather", "foster", "family", "clan",
            "from the same womb", "childless", "firstborn",
            "father-in-law", "mother-in-law", "son-in-law", "daughter-in-law",
        ],
    },
    "marriage": {
        "prompts": [
            "marriage, weddings, husbands, wives, and betrothal",
            "taking a spouse, a dowry, or a bride in matrimony",
        ],
        "anti_prompts": [
            "joining, fastening, or yoking of objects or animals together",
        ],
        "seeds": [
            "marriage", "married", "unmarried", "wedding", "bride",
            "bridegroom", "wife", "husband", "spouse", "betroth", "dowry",
            "wedlock", "concubine", "bachelor", "celibacy", "divorce",
            "wooing", "suitor",
        ],
    },
    "friendship_affection": {
        "prompts": [
            "close friendship and companionship between people",
            "affection, tenderness, warmth, and love felt for another person",
            "goodwill, kindness, and devotion shown to a friend",
        ],
        "anti_prompts": [
            "love of money, honour, power, wisdom, or pleasure",
            "friendly to a doctrine, party, cause, or place",
            "mild, kindly, or favourable weather or climate",
        ],
        "seeds": [
            "friend", "friendship", "companion", "comrade", "beloved",
            "darling", "affection", "loving", "love of", "dear to",
            "fond of", "brotherly love", "goodwill", "benevolent",
        ],
    },
    "hostility_enmity": {
        "prompts": [
            "hostility, enmity, hatred, and being someone's enemy",
            "envy, jealousy, grudges, quarrels, and rivalry between people",
        ],
        "anti_prompts": [
            "a harsh taste, bitter flavour, or rough physical texture",
            "stormy, violent, or savage weather or landscape",
            "the difficulty, harshness, or severity of a task",
        ],
        "seeds": [
            "enemy", "enmity", "hostile", "hostility", "hatred", "hate",
            "foe", "quarrel", "feud", "grudge", "rival", "rivalry",
            "envy", "jealous", "wrath", "revenge", "vengeance", "strife",
            "adversary", "insolence", "insolent", "arrogance", "abuse",
            "reproach", "insult",
        ],
    },
    "hospitality_xenia": {
        "prompts": [
            "hospitality, guest-friendship, and welcoming guests and strangers",
            "a host entertaining a guest at table or in the home",
        ],
        "anti_prompts": [
            "a mercenary or hired foreign soldier",
            "a foreign, strange, or unfamiliar thing in general",
        ],
        "seeds": [
            "guest", "host", "hospitality", "stranger", "guest-friend",
            "xenia", "welcome", "entertain", "banquet", "feast",
            "table-companion", "inhospitable",
        ],
    },
    "erotic_pederasty": {
        "prompts": [
            "romantic or sexual love between lovers",
            "a lover, beloved, or object of erotic desire",
            "courtesans, mistresses, and pederastic relationships",
        ],
        "anti_prompts": [
            "abstract desire for wealth, power, fame, or victory",
            "love of wisdom, virtue, or learning",
        ],
        "seeds": [
            "lover", "beloved boy", "pederasty", "erotic", "courtesan",
            "paramour", "mistress", "sexual love", "desire for", "wooer",
            "eromenos", "erastes", "hetaira",
        ],
    },
    "rank_kingship_rule": {
        "prompts": [
            "kingship, royalty, rulers, tyrants, and political authority",
            "social rank, nobility, high or low birth, and aristocracy",
        ],
        "anti_prompts": [
            "mastery of a craft, art, skill, or subject",
            "a rule, principle, standard, or measuring instrument",
            "the first principle, origin, or beginning of something",
            "self-control or command over one's own passions",
        ],
        "seeds": [
            "king", "kingship", "queen", "royal", "ruler", "rule over",
            "reign", "sovereign", "tyrant", "prince", "lord", "lordly",
            "master", "chief", "chieftain", "leader", "leading",
            "magistrate", "archon", "governor", "noble", "nobility",
            "high-born", "well-born", "low-born", "ignoble", "aristocrat",
            "commoner", "rank", "throne", "sceptre",
            "not ruled by a king", "subject to", "dominion", "authority",
        ],
    },
    "military_rank": {
        "prompts": [
            "military ranks, commanders, generals, and leading troops",
            "soldiers, an army, and command in war",
        ],
        "anti_prompts": [
            "arrangement, order, or array of objects or words",
            "bravery, boldness, or excellence in a non-military moral sense",
        ],
        "seeds": [
            "general", "commander", "captain", "officer", "soldier",
            "troop", "army", "phalanx", "garrison", "recruit", "levy",
            "muster", "host-leading", "leader of the host",
            "comrade in arms", "brave", "valiant",
        ],
    },
    "servitude_slavery": {
        "prompts": [
            "slavery, servants, servitude, and forced labour",
            "a slave, freedman, or bondservant serving a master",
        ],
        "anti_prompts": [
            "subjection to a law, passion, necessity, or fate",
            "a tool or thing that serves a mechanical function",
        ],
        "seeds": [
            "slave", "slavery", "servant", "serf", "servitude", "bondage",
            "freedman", "manumission", "handmaid", "attendant",
            "hired labourer", "hireling", "impress", "impressment",
            "forced service",
        ],
    },
    "messenger_herald_envoy": {
        "prompts": [
            "messengers, heralds, envoys, and ambassadors between parties",
            "a herald or courier carrying tidings or a dispatch",
        ],
        "anti_prompts": [
            "a message, sign, or announcement in a purely abstract sense",
            "a proclamation or invocation in a religious rite",
        ],
        "seeds": [
            "messenger", "envoy", "herald", "ambassador", "courier",
            "message", "tidings", "dispatch",
        ],
    },
    "civic_community": {
        "prompts": [
            "civic community, citizenship, tribes, and organized groups of people",
            "a citizen or member of a city, tribe, or deme",
        ],
        "anti_prompts": [
            "a gregarious, herding, or social animal",
            "common, shared, or public property or resources",
            "a biological or natural community of organisms",
        ],
        "seeds": [
            "citizen", "citizenship", "fellow-citizen", "assembly",
            "community", "tribe", "deme", "gregarious", "social",
            "band of youths", "age-class", "member of",
        ],
    },
}


GREEK_RANGES = ((0x0370, 0x03FF), (0x1F00, 0x1FFF))


def is_mostly_greek(text: str, ratio: float = 0.5) -> bool:
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return False
    greek = sum(1 for c in letters
                if any(lo <= ord(c) <= hi for lo, hi in GREEK_RANGES))
    return greek / len(letters) >= ratio


BOLD_RE = re.compile(r"\*\*(.+?)\*\*", re.S)


def extract_gloss(entry: str) -> str:
    spans = []
    for m in BOLD_RE.finditer(entry):
        chunk = m.group(1).strip()
        if not chunk or is_mostly_greek(chunk):
            continue
        spans.append(chunk)
    gloss = "; ".join(dict.fromkeys(spans))
    gloss = re.sub(r"\s+", " ", gloss).strip("; ").strip()
    return gloss


def split_senses(gloss: str):
    """Split a gloss into its separate senses on ';' so each can be scored alone."""
    senses = [s.strip() for s in gloss.split(";") if s.strip()]
    return senses or [gloss]


def parse_cex(path: Path, limit: int = 0):
    """Yield (seq, urn, lemma, entry) from CEX; stop after *limit* rows if > 0."""
    count = 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split("#", 3)
            if len(parts) < 4:
                continue
            seq, urn, lemma, entry = parts
            if not urn.startswith("urn:"):
                continue
            yield seq, urn, lemma.strip("· "), entry
            count += 1
            if limit and count >= limit:
                break


def compile_seed_patterns():
    patterns = {}
    for cat, spec in CATEGORIES.items():
        alts = sorted(spec["seeds"], key=len, reverse=True)
        pat = r"\b(?:" + "|".join(re.escape(s) for s in alts) + r")\b"
        patterns[cat] = re.compile(pat, re.IGNORECASE)
    return patterns



def main():
    if not CEX_PATH.exists():
        sys.exit(f"ERROR: CEX file not found:\n  {CEX_PATH}")

    import numpy as np

    t0 = time.time()

    print(f"\n[1/5] Parsing {CEX_PATH.name} ...")
    records = []
    for seq, urn, lemma, entry in tqdm(
            parse_cex(CEX_PATH, limit=TEST_LIMIT),
            desc="  parsing", unit=" entries"):
        gloss = extract_gloss(entry)
        if gloss:
            records.append({
                "seq": seq, "urn": urn, "lemma": lemma,
                "gloss": gloss, "entry_len": len(entry),
            })
    if not records:
        sys.exit("No entries with an English gloss were found — check the CEX file.")
    print(f"  → {len(records):,} entries with an English gloss  "
          f"({time.time()-t0:.1f}s)")

    cat_names = list(CATEGORIES.keys())
    n_cat = len(cat_names)
    n_ent = len(records)

   
    print("\n[2/5] Seed-keyword pass ...")
    seed_patterns = compile_seed_patterns()
    seed_hit = np.zeros((n_ent, n_cat), dtype=bool)
    for i, rec in enumerate(records):
        for ci, cat in enumerate(cat_names):
            if seed_patterns[cat].search(rec["gloss"]):
                seed_hit[i, ci] = True
    print(f"  → {int(seed_hit.any(axis=1).sum()):,} entries touched at least one seed")

    chunk_texts, entry_starts = [], []
    for rec in records:
        entry_starts.append(len(chunk_texts))
        chunk_texts.extend(split_senses(rec["gloss"]))
    entry_starts = np.asarray(entry_starts, dtype=np.int64)
    print(f"  → {len(chunk_texts):,} individual senses to score")


    print(f"\n[3/5] Loading model '{EMBED_MODEL}' ...")
    t2 = time.time()
    from sentence_transformers import SentenceTransformer
    try:
        from huggingface_hub import snapshot_download
        local_path = snapshot_download(
            repo_id=f"sentence-transformers/{EMBED_MODEL}",
            token=hf_token or None,
            ignore_patterns=["additional_chat_templates*", "*.msgpack",
                             "flax_model*", "tf_model*", "rust_model*"],
        )
        model = SentenceTransformer(local_path)
    except Exception as e:
        print(f"  snapshot_download failed ({e}); trying direct load ...")
        model = SentenceTransformer(f"sentence-transformers/{EMBED_MODEL}")
    print(f"  Model loaded ({time.time()-t2:.1f}s)")

    global_anti = model.encode(GLOBAL_ANTI_PROMPTS, normalize_embeddings=True,
                               show_progress_bar=False)
    pos_mats, neg_mats = {}, {}
    for cat, spec in CATEGORIES.items():
        pos_mats[cat] = model.encode(spec["prompts"], normalize_embeddings=True,
                                     show_progress_bar=False)
        cat_anti = spec.get("anti_prompts", [])
        neg_texts = model.encode(cat_anti, normalize_embeddings=True,
                                 show_progress_bar=False) if cat_anti else None
        neg_mats[cat] = (np.vstack([neg_texts, global_anti])
                         if neg_texts is not None else global_anti)

    print(f"\n[4/5] Embedding {len(chunk_texts):,} senses "
          f"(one-time; use TEST_LIMIT for a quick check) ...")
    chunk_vecs = model.encode(chunk_texts, normalize_embeddings=True,
                              batch_size=256, show_progress_bar=True)
    chunk_vecs = np.asarray(chunk_vecs, dtype=np.float32)


    print("\n[5/5] Scoring & filtering ...")
    entry_pos = np.zeros((n_ent, n_cat), dtype=np.float32)
    entry_neg = np.zeros((n_ent, n_cat), dtype=np.float32)
    assigned  = np.zeros((n_ent, n_cat), dtype=bool)   # passed the full test
    base_pass = np.zeros((n_ent, n_cat), dtype=bool)   # passed WITHOUT needing a seed

    for ci, cat in enumerate(cat_names):
        pos_chunk = (chunk_vecs @ pos_mats[cat].T).max(axis=1)   # (n_chunks,)
        neg_chunk = (chunk_vecs @ neg_mats[cat].T).max(axis=1)
        margin_ok = (pos_chunk - neg_chunk) >= MARGIN
        base_ok   = pos_chunk >= THRESHOLD
        seed_ok   = pos_chunk >= SEED_CONFIRM_THRESHOLD

        q_base = (margin_ok & base_ok).astype(np.int8)
        q_seed = (margin_ok & seed_ok).astype(np.int8)

        # any-sense-qualifies, per entry (segments are contiguous per entry)
        ent_base = np.maximum.reduceat(q_base, entry_starts).astype(bool)
        ent_seed = np.maximum.reduceat(q_seed, entry_starts).astype(bool)
        entry_pos[:, ci] = np.maximum.reduceat(pos_chunk, entry_starts)
        entry_neg[:, ci] = np.maximum.reduceat(neg_chunk, entry_starts)

        base_pass[:, ci] = ent_base
        assigned[:, ci]  = ent_base | (seed_hit[:, ci] & ent_seed)

    output_rows, score_rows = [], []
    for i, rec in enumerate(records):
        cand = np.where(assigned[i])[0]
        if cand.size:
            best = cand[np.argmax(entry_pos[i, cand])]
            if MULTI_LABEL:
                keep = cand[entry_pos[i, best] - entry_pos[i, cand] <= TIE_DELTA]
            else:
                keep = [best]
            for ci in keep:
                if base_pass[i, ci] and seed_hit[i, ci]:
                    method = "seed+embed"
                elif base_pass[i, ci]:
                    method = "embed"
                else:
                    method = "seed"
                output_rows.append({
                    "lemma":     rec["lemma"],
                    "urn":       rec["urn"],
                    "category":  cat_names[ci],
                    "method":    method,
                    "pos_score": round(float(entry_pos[i, ci]), 3),
                    "neg_score": round(float(entry_neg[i, ci]), 3),
                    "margin":    round(float(entry_pos[i, ci] - entry_neg[i, ci]), 3),
                    "entry_len": rec["entry_len"],
                    "gloss":     rec["gloss"],
                })

        best_any = int(np.argmax(entry_pos[i]))
        score_rows.append({
            "lemma": rec["lemma"], "gloss": rec["gloss"],
            "best_category": cat_names[best_any],
            "best_pos": round(float(entry_pos[i, best_any]), 3),
            "best_neg": round(float(entry_neg[i, best_any]), 3),
            "assigned": bool(assigned[i].any()),
            "entry_len": rec["entry_len"],
            **{f"pos_{c}": round(float(entry_pos[i, k]), 3)
               for k, c in enumerate(cat_names)},
        })


    if output_rows:
        df = (pd.DataFrame(output_rows)
                .sort_values(["category", "pos_score"], ascending=[True, False]))
    else:
        df = pd.DataFrame(columns=["lemma", "urn", "category", "method",
                                   "pos_score", "neg_score", "margin",
                                   "entry_len", "gloss"])
    df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig",
              quoting=csv.QUOTE_MINIMAL)
    pd.DataFrame(score_rows).to_csv(OUT_SCORES, index=False, encoding="utf-8-sig")

    total = time.time() - t0
    print(f"\n✓ Done in {total/60:.1f} min")
    print(f"  {len(df):,} assignments  "
          f"({df['lemma'].nunique():,} unique lemmata)"
          if len(df) else "  0 assignments — loosen THRESHOLD / MARGIN.")
    print(f"  Main CSV  → {OUT_PATH}")
    print(f"  Scores CSV→ {OUT_SCORES}")
    if len(df):
        print("\nEntries per category:")
        print(df["category"].value_counts().to_string())


if __name__ == "__main__":
    main()


[1/5] Parsing lsj_chicago.cex ...


  parsing: 0 entries [00:00, ? entries/s]

  parsing: 116854 entries [00:09, 12149.06 entries/s]


  → 91,358 entries with an English gloss  (9.6s)

[2/5] Seed-keyword pass ...
  → 4,483 entries touched at least one seed
  → 225,187 individual senses to score

[3/5] Loading model 'all-MiniLM-L6-v2' ...


Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

  Model loaded (0.6s)

[4/5] Embedding 225,187 senses (one-time; use TEST_LIMIT for a quick check) ...


Batches:   0%|          | 0/880 [00:00<?, ?it/s]


[5/5] Scoring & filtering ...

✓ Done in 18.8 min
  2,122 assignments  (2,122 unique lemmata)
  Main CSV  → lsj_social_relations.csv
  Scores CSV→ lsj_social_relations_scores.csv

Entries per category:
category
kinship_family            367
hostility_enmity          342
marriage                  297
military_rank             205
rank_kingship_rule        202
servitude_slavery         186
friendship_affection      164
hospitality_xenia         128
messenger_herald_envoy     97
civic_community            91
erotic_pederasty           43
